# POC — Extração de Dados de Documentos com IA Generativa
### Desafio Técnico — Pesquisador(a) em IA Generativa (P&D)
**Candidato:** Danilo &nbsp;|&nbsp; **Data:** 1 de julho de 2026

Este notebook acompanha o **Dossiê de Pesquisa** (documento técnico em PDF) e demonstra:

1. **A implementação da técnica recomendada** (Seção 5 do Dossiê): extração multimodal em passo único com *Structured Outputs*, via Claude API — código completo e correto em `src/extrator_single_pass.py`.
2. **Um baseline executável de ponta a ponta neste ambiente**, sem depender de nenhuma chave de API: OCR clássico local (Tesseract), em `src/baseline_ocr_local.py` — é o que gerou os resultados empíricos da Seção 4.6 do Dossiê.

> **Por que dois pipelines?** O ambiente usado para desenvolver este desafio não tem GPU nem uma `ANTHROPIC_API_KEY` configurada (nem de qualquer outro provedor). A Seção 4.1 do Dossiê detalha essa limitação. O código da Parte 1 está pronto para produção — só falta uma credencial, que quem avalia este desafio pode fornecer para ver o pipeline recomendado rodando de verdade.


## Estrutura do repositório

```
poc/
├── schemas/                        # JSON Schemas dos 3 casos de uso
│   ├── schema_cnh.json
│   ├── schema_fatura.json
│   └── schema_documento_extenso.json
├── samples/                        # Os 3 documentos de exemplo do desafio
│   ├── cnh_exemplo.jpeg
│   ├── fatura_energia.jpg
│   └── claude3_model_family.pdf
├── src/
│   ├── extrator_single_pass.py     # Técnica A (recomendada) — precisa de API key
│   └── baseline_ocr_local.py       # Baseline local — roda sem nenhuma credencial
├── outputs/                        # Resultados salvos em JSON
└── notebooks/POC.ipynb             # Este notebook
```


In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))
os.chdir("..")  # roda a partir da raiz do projeto poc/, para os caminhos relativos dos schemas/samples funcionarem
print("Diretorio de trabalho:", os.getcwd())


Diretorio de trabalho: /home/claude/poc


---
## Parte 1 — Técnica recomendada: extração em passo único (Structured Outputs)

`src/extrator_single_pass.py` implementa a arquitetura recomendada no Dossiê (Seção 3.1 e 5): uma **única chamada** à Claude API recebe o documento (imagem ou PDF nativo) + um JSON Schema estrito (`strict: true`), e devolve diretamente o objeto já validado contra o schema do cliente — sem uma segunda chamada de formatação.


In [2]:
from src.extrator_single_pass import extrair_documento, executar_todos_os_casos, CASOS, MODELO_PADRAO

print("Modelo padrao configurado:", MODELO_PADRAO)
print()
print("Casos de uso definidos:")
for nome, cfg in CASOS.items():
    print(f"  - {nome}: arquivo={cfg['caminho_arquivo']}, schema={cfg['caminho_schema']}")


Modelo padrao configurado: claude-haiku-4-5

Casos de uso definidos:
  - cnh: arquivo=samples/cnh_exemplo.jpeg, schema=schemas/schema_cnh.json
  - fatura: arquivo=samples/fatura_energia.jpg, schema=schemas/schema_fatura.json
  - documento_extenso: arquivo=samples/claude3_model_family.pdf, schema=schemas/schema_documento_extenso.json


### Executando os 3 casos

Se a variável de ambiente `ANTHROPIC_API_KEY` estiver definida, a célula abaixo faz **chamadas reais** à Claude API e retorna o JSON extraído de cada documento, pronto no schema do cliente — o resultado esperado em produção.

Neste ambiente de desenvolvimento (sem chave configurada), a chamada falha de forma controlada na etapa de autenticação — o que confirma que o restante do pipeline (montagem do payload multimodal, codificação do documento, definição do schema estrito) está correto e é só isso que falta.


In [3]:
if not os.environ.get("ANTHROPIC_API_KEY"):
    print("[AVISO] ANTHROPIC_API_KEY nao definida neste ambiente.")
    print("As chamadas abaixo vao tentar de verdade e devem falhar apenas na autenticacao.")
    print("Para rodar de ponta a ponta: export ANTHROPIC_API_KEY='sk-ant-...' antes de iniciar o kernel.\n")

resultados_tecnica_a = executar_todos_os_casos()

for r in resultados_tecnica_a:
    print("=" * 70)
    print(f"Caso: {r.caso} | modelo: {r.modelo} | ok={r.ok}")
    if r.ok:
        print(f"Latencia: {r.latencia_segundos}s | tokens_in={r.tokens_entrada} | tokens_out={r.tokens_saida}")
        print(json.dumps(r.dados, ensure_ascii=False, indent=2))
    else:
        print(f"Erro (esperado sem API key): {r.erro}")


[AVISO] ANTHROPIC_API_KEY nao definida neste ambiente.
As chamadas abaixo vao tentar de verdade e devem falhar apenas na autenticacao.
Para rodar de ponta a ponta: export ANTHROPIC_API_KEY='sk-ant-...' antes de iniciar o kernel.



Caso: cnh | modelo: claude-haiku-4-5 | ok=False
Erro (esperado sem API key): TypeError: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"
Caso: fatura | modelo: claude-haiku-4-5 | ok=False
Erro (esperado sem API key): TypeError: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"
Caso: documento_extenso | modelo: claude-haiku-4-5 | ok=False
Erro (esperado sem API key): TypeError: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"


**Leitura do resultado acima:** o `TypeError` de autenticação confirma que o código chegou até a tentativa de chamada de rede — ou seja, toda a lógica anterior (carregar o schema, ler o arquivo, codificar em base64, montar o content-block de imagem/PDF, montar a tool call com `strict: true`) executou sem erros. É exatamente o comportamento esperado e documentado na Seção 4.1 do Dossiê.


---
## Parte 2 — Baseline executável: OCR clássico local

`src/baseline_ocr_local.py` **roda de ponta a ponta neste ambiente**, sem GPU e sem nenhuma chave de API — usa apenas Tesseract (motor de OCR open-source) e `pdf2image`. Os números abaixo são **reais**, medidos agora, neste sandbox — não são estimativas.

Este baseline representa o primeiro estágio típico de um pipeline OCR-primeiro (o tipo de abordagem descartada na Seção 2.2 do Dossiê), e serve para demonstrar empiricamente, com os 3 documentos reais do desafio, as limitações discutidas na Seção 4.6.


In [4]:
from src.baseline_ocr_local import rodar_baseline

resultados_ocr = rodar_baseline(diretorio_samples="samples")

for r in resultados_ocr:
    print("=" * 70)
    print(f"Caso: {r.caso} | arquivo: {r.arquivo}")
    print(f"Tempo: {r.tempo_segundos}s | Caracteres extraidos: {r.n_caracteres_extraidos}")
    print("--- amostra do texto bruto (sem estrutura) ---")
    print(r.amostra_texto)
    print()


Caso: cnh | arquivo: cnh_exemplo.jpeg
Tempo: 0.555s | Caracteres extraidos: 11
--- amostra do texto bruto (sem estrutura) ---
E 000000000

Caso: fatura | arquivo: fatura_energia.jpg
Tempo: 3.087s | Caracteres extraidos: 1777
--- amostra do texto bruto (sem estrutura) ---
Grupo Neenergia

n celpe

DADOS DO CLIENTE
MARIA JOSÉ DA SILVA

cpr: 123458 78940

CLASSIFICAÇÃO

BI RESIDENCIAL
RESIDENCIAL

Monotásico

NDANOTAFISCAL | SERIE
conoogas ÚNICA
APRESENTAÇÃO | Nº DO cuENTE

Consumo ato gut

Tarifa Social de Energia Elétrica: Criada pela Lei 10,438, de 26/04/02.
NOTA FISCAL + FATURA = CONTA DE ENERGIA ELÉTRICA

Companhia Energélica de Perempuco.

Ay, João de Batos 11, Boa Vista Recle, Pemêmibuco - CEP 50050-602

CNPJ 10.835 98

43-93 | vm ele com be

ENDEREÇO DA UNIDADE CONSUMIDORA
RUA JOÃO FERNANDES VIEIRA 175

BOAVISTA.

sosso-150

conta contrato

1234567890

) 

Caso: documento_extenso_pg8_tabela | arquivo: claude3_model_family.pdf (pg. 8)
Tempo: 1.916s | Caracteres extraidos: 1131
--- 

### O que os resultados acima mostram

| Documento | Achado |
|---|---|
| **CNH** | Apenas 11 caracteres extraídos (`"E 000000000"`) — o padrão de segurança do fundo (guilloché) quebra o OCR clássico. Nenhum nome, CPF ou data foi lido. |
| **Fatura de energia** | `"CPF: 123.456.789-10"` virou `"cpr: 123458 78940"`; cabeçalhos de tabela ficaram colados (`"NDANOTAFISCAL \| SERIE"`) — a grade da tabela não é reconstruída. |
| **Paper, pág. 8 (tabela)** | Dígitos trocados: `"67.5%"` → `"615%"`, `"57.1%"` → `"STI%"` — um erro silencioso, sem sinalização. |
| **Paper, pág. 9 (gráfico)** | 0% dos dados do gráfico de barras foi capturado — OCR clássico não interpreta imagens não-textuais. |

Discussão completa na **Seção 4.6 do Dossiê de Pesquisa**.


In [5]:
# Salva os resultados brutos em JSON (mesmo arquivo referenciado no Dossie, Secao 4.6)
os.makedirs("outputs", exist_ok=True)
with open("outputs/baseline_ocr_resultados.json", "w", encoding="utf-8") as f:
    json.dump([r.__dict__ for r in resultados_ocr], f, ensure_ascii=False, indent=2)
print("Salvo em outputs/baseline_ocr_resultados.json")


Salvo em outputs/baseline_ocr_resultados.json


---
## Conclusão

- A **Parte 1** contém a implementação de referência da técnica recomendada no Dossiê — correta e pronta para produção, faltando apenas uma credencial de API que este ambiente de desenvolvimento não possui.
- A **Parte 2** roda de ponta a ponta neste sandbox e gerou evidência empírica real das limitações de um pipeline OCR-primeiro nos 3 documentos do desafio, sustentando os achados da Seção 4.6 do Dossiê.

Para a análise completa de custo, latência, acurácia (benchmarks de terceiros) e a recomendação final com roadmap, ver o **Dossiê de Pesquisa** (`Dossie_Pesquisa_IA_Generativa.pdf`).
